# Horizon perturbations from antenna-position error

Baseline horizon elevation profile $\alpha_h(\mathrm{az})$ (panel a) and the
change $\Delta\alpha_h = \alpha_h^{\mathrm{shift}} - \alpha_h^{\mathrm{nominal}}$
for +1 m antenna displacements East / North / Up (panel b). A 1 m move changes
the horizon by $\lesssim 0.1^\circ$ over most azimuths, spiking to $\sim 1^\circ$
only at steep cliff edges (where a lateral move slides a near-vertical horizon
edge sideways); raising the antenna (Up +1 m) lowers the horizon by a
near-uniform small offset.

Data: `horizon_perturbations.npz` (keys `names`, `az_grid`, `alpha_h`).
Produces `horizon_perturbations_1col.pdf` (the single-column figure used in the
paper) and the wider `horizon_perturbations.pdf`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
d = np.load("horizon_perturbations.npz", allow_pickle=True)
names = [str(n) for n in d["names"]]
az = np.degrees(d["az_grid"])          # azimuth grid [deg]
alpha = np.degrees(d["alpha_h"])       # horizon elevation per scenario [deg]
base = alpha[names.index("nominal")]   # baseline (nominal-position) profile

# +1 m shift directions, with colour-blind-safe (Okabe-Ito) colours that
# stay clear of the orange baseline fill.
SHIFTS = [
    ("x_p_1", "East +1 m", "#0072B2"),   # blue
    ("y_p_1", "North +1 m", "#CC79A7"),  # reddish purple
    ("z_p_1", "Up +1 m", "#009E73"),     # bluish green
]
FILL = "#c56a39"

In [ ]:
def build_figure(az, alpha, names, base, figsize, legend_kw,
                 resid_ylim=(-1.3, 1.3)):
    """Stacked baseline (a) / shift-residual (b) horizon figure.

    ``resid_ylim`` sets the residual-panel limits; the narrow single-column
    variant extends the range to make headroom for the in-panel legend.
    """
    fig, (axt, axb) = plt.subplots(
        2, 1, figsize=figsize, sharex=True,
        gridspec_kw=dict(height_ratios=[3, 1.4]), layout="constrained",
    )

    axt.fill_between(az, 0, base, color=FILL, lw=0)
    axt.plot(az, base, color="black", lw=1.1)
    axt.set_ylabel("Horizon Angle [deg]")
    axt.set_ylim(0, 40)
    axt.set_axisbelow(False)  # gridlines on top of the opaque fill

    for tag, lbl, c in SHIFTS:
        axb.plot(az, alpha[names.index(tag)] - base, color=c, lw=1.0, label=lbl)
    axb.axhline(0, color="0.6", lw=0.7, ls="--")
    axb.set_ylabel(r"$\Delta$ Horizon [deg]")
    axb.set_xlabel("Azimuthal Angle [deg]")
    axb.set_ylim(*resid_ylim)
    axb.legend(**legend_kw)

    for ax, tag in ((axt, "(a)"), (axb, "(b)")):
        ax.set_xlim(0, 360)
        ax.grid(alpha=0.3)
        ax.text(0.012, 0.93, tag, transform=ax.transAxes, va="top")
    return fig

In [ ]:
# Wide variant: in-panel 3-column legend.
with plt.rc_context({"font.size": 10}):
    fig = build_figure(
        az, alpha, names, base, figsize=(6.5, 4.8),
        legend_kw=dict(ncol=3, loc="upper right", fontsize=8,
                       columnspacing=1.0, handlelength=1.6),
    )
    fig.savefig("horizon_perturbations.pdf")

# Single-column variant used in the paper: smaller fonts, legend in panel (b)
# with the residual range extended for headroom above the curves.
with plt.rc_context({"font.size": 8}):
    fig = build_figure(
        az, alpha, names, base, figsize=(3.4, 4.0),
        legend_kw=dict(ncol=3, loc="upper center", fontsize=6.5,
                       columnspacing=0.8, handlelength=1.2, handletextpad=0.4),
        resid_ylim=(-2.1, 2.1),
    )
    fig.savefig("horizon_perturbations_1col.pdf")